<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 04. Prioritized Replay

## Prioritized Experience Replay

- **원 논문:** [Prioritized Experience Replay](https://arxiv.org/abs/1511.05952)
- **저자 / 발표:** Tom Schaul, John Quan, Ioannis Antonoglou, and David Silver · ICLR (2016)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** DQN, TD error, importance sampling
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** TD-error proportional sampling과 annealed importance weights로 로컬 transition Q-table을 학습한다.

**축소하거나 생략한 부분:** 논문은 Atari replay trees와 Double DQN을 사용한다. 축소판은 모든 transition을 메모리에 두고 정확한 categorical probabilities를 계산한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.2 | TD-error priority | 큰 surprise를 가진 transition을 더 자주 replay한다. |
| §3.3, Eq. (1) | proportional sampling | P(i)=p_i^alpha/sum p^alpha와 p=|delta|+epsilon을 구현한다. |
| §3.4 and Algorithm 1 | importance weights and prioritized Double DQN | w_i=(N P(i))^-beta를 정규화해 weighted TD loss를 만든다. |

## 핵심 재현

        Eq. (1)로 TD error가 큰 전이를 자주 뽑되, §3.4의 importance weight로 바뀐 sampling
        distribution의 편향을 완화합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,np.random.seed,Path,json.loads,Path.read_text,torch.tensor,torch.float32,F.one_hot,torch.as_tensor,torch.zeros,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 11개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

<details>
<summary><code>json.loads(s, ...)</code></summary>

#### `json.loads(s, ...)`

- **역할:** JSON 문자열을 Python 객체로 역직렬화합니다.
- **입력·반환:** 문자열·bytes를 받고 dict·list 등의 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 메모리에 있는 JSON line이나 응답을 구조화합니다.
- **주의점:** 입력이 한 문서인지 JSON Lines의 한 줄인지 구분해야 합니다.
- **공식 문서:** [json.loads](https://docs.python.org/3/library/json.html#json.loads)

</details>

<details>
<summary><code>path.read_text(encoding=None, errors=None)</code></summary>

#### `path.read_text(encoding=None, errors=None)`

- **역할:** 텍스트 파일 전체를 문자열로 읽습니다.
- **입력·반환:** 인코딩 설정을 받고 문자열을 반환합니다.
- **이 셀에서 쓰는 이유:** 작은 설정·말뭉치·메타데이터 파일을 명시적 인코딩으로 읽습니다.
- **주의점:** 학습용 대용량 파일에는 전체 메모리 적재가 부담될 수 있습니다.
- **공식 문서:** [Path.read_text](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>F.one_hot(tensor, num_classes=-1)</code></summary>

#### `F.one_hot(tensor, num_classes=-1)`

- **역할:** 정수 class ID를 one-hot 텐서로 변환합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
H_{i,c}=\mathbb{1}[x_i=c]
$$

- **기호:** $x_i$는 class ID이고 $H_{i,c}$는 ID가 $c$일 때만 1인 indicator입니다.
- **수식 → 코드:** API가 정수 index를 class 축이 추가된 indicator tensor로 바꿉니다.
- **직관:** 범주 번호를 거리 있는 수로 보지 않고 서로 독립적인 축으로 표현합니다.
- **shape 확인:** 입력 `[...] -> [...,C]`; 출력은 정수 dtype이므로 loss에 따라 float 변환이 필요합니다.
- **공식 문서:** [F.one_hot](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)

</details>

<details>
<summary><code>torch.as_tensor(data, dtype=None, device=None)</code></summary>

#### `torch.as_tensor(data, dtype=None, device=None)`

- **역할:** 가능하면 데이터를 공유해 텐서로 해석합니다.
- **입력·반환:** 배열형 데이터를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 불필요한 복사를 줄이며 입력을 텐서 계약으로 맞춥니다.
- **주의점:** 원본과 저장공간을 공유할 수 있어 변경 부작용을 확인해야 합니다.
- **공식 문서:** [torch.as_tensor](https://docs.pytorch.org/docs/stable/generated/torch.as_tensor.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(441)
np.random.seed(441)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
P(i)=\frac{p_i^\alpha}{\sum_kp_k^\alpha},\qquad w_i=\left(NP(i)\right)^{-\beta}/\max_jw_j
$$

- **기호와 역할:** $p_i=|\delta_i|+\epsilon$은 priority, $\alpha$는 prioritization, $\beta$는 importance correction 강도입니다.
- **shape/state 계약:** `priority/probability/weight [N_replay], sampled indices [B], TD error [B]`
- **논문 위치:** `§3.2`의 **TD-error priority**
- **코드 Task:** 비균등 sampling, IS-weighted TD loss와 priority 갱신을 구현합니다.
- **학습·평가 흐름:** priority → categorical sample → weighted TD update → |TD| priority → 빈도 평가
- **원문 대비 한계:** 논문은 Atari replay trees와 Double DQN을 사용한다. 축소판은 모든 transition을 메모리에 두고 정확한 categorical probabilities를 계산한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.sum,torch.no_grad,Optimizer.zero_grad,Tensor.backward,Optimizer.step,Tensor.detach,F.mse_loss -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>F.mse_loss(input, target, reduction=&#x27;mean&#x27;)</code></summary>

#### `F.mse_loss(input, target, reduction='mean')`

- **역할:** 예측과 target의 제곱 오차를 계산합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2
$$

- **기호:** $\hat y_i$는 예측, $y_i$는 target입니다.
- **수식 → 코드:** `reduction='mean'`이 모든 원소의 제곱 오차를 평균냅니다.
- **직관:** 큰 오차를 제곱해 더 강하게 벌점하므로 회귀와 reconstruction에 자주 씁니다.
- **shape 확인:** 예측과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.mse_loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class PrioritizedReplayAgent(nn.Module):
    """Prioritized Replay의 핵심 학습·평가 경로. 입출력 계약: priority/probability/weight
    [N_replay], sampled indices [B], TD error [B]."""

    def __init__(self, q_table, optimizer, priorities, discount, alpha=0.7):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.q_table = q_table
        self.optimizer = optimizer
        self.priorities = priorities
        self.discount = discount
        self.alpha = alpha

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"replay_size": len(self.priorities), "alpha": self.alpha}

    def forward(self, state, action):
        """policy/value/Q 순전파를 계산한다. shape 계약: priority/probability/weight [N_replay],
        sampled indices [B], TD error [B]."""
        return self.q_table[state, action]

    def sampling_probabilities(self):
        """priority를 정규화한 replay sampling 확률을 반환한다."""
        scaled_priority = self.priorities.clamp_min(1e-06).pow(self.alpha)
        return scaled_priority / scaled_priority.sum()

    def loss(self, indices, beta):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        predicted_q = self.forward(states[indices], actions[indices])
        with torch.no_grad():
            next_q = self.q_table[next_states[indices]].max(dim=1).values
            target_q = rewards[indices]
            target_q = target_q + self.discount * (1.0 - dones[indices]) * next_q
        td_error = target_q - predicted_q
        probability = self.sampling_probabilities()[indices]
        raw_weight = (len(self.priorities) * probability).pow(-beta)
        importance_weight = raw_weight / raw_weight.max()
        objective = (importance_weight * td_error.square()).mean()
        return (objective, td_error)

    def update(self, indices, beta):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective, td_error = self.loss(indices, beta)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        self.priorities[indices] = td_error.detach().abs() + 0.001
        return float(objective.detach())

    def evaluate(self, reference_q):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            mse = F.mse_loss(self.q_table, reference_q)
            probability = self.sampling_probabilities()
        return {"q_mse": float(mse), "probability_sum": float(probability.sum())}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3.3, Eq. (1)` — **proportional sampling**
- **구현 이유:** P(i)=p_i^alpha/sum p^alpha와 p=|delta|+epsilon을 구현한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def per_probabilities(priority, alpha=0.7):
    scaled = priority.clamp_min(1e-6).pow(alpha)
    return scaled / scaled.sum()


def is_weights(probability, beta=0.5):
    raw = (len(probability) * probability).pow(-beta)
    return raw / raw.max()


p = per_probabilities(torch.tensor([1.0, 2.0, 4.0]))
w = is_weights(p)
assert torch.allclose(p.sum(), torch.tensor(1.0)) and torch.allclose(
    w.max(), torch.tensor(1.0)
)

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.4 and Algorithm 1` — **importance weights and prioritized Double DQN**
- **구현 이유:** w_i=(N P(i))^-beta를 정규화해 weighted TD loss를 만든다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Parameter,torch.optim.Adam,torch.ones,np.random.default_rng,np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Parameter(data, requires_grad=True)</code></summary>

#### `nn.Parameter(data, requires_grad=True)`

- **역할:** 텐서를 학습 파라미터로 등록합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Parameter](https://docs.pytorch.org/docs/stable/generated/torch.nn.Parameter.html)

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>np.random.default_rng(seed=None)</code></summary>

#### `np.random.default_rng(seed=None)`

- **역할:** 독립적인 NumPy 난수 생성기 `Generator`를 만듭니다.
- **입력·반환:** 선택적 시드를 받고 `Generator` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 전역 상태를 오염시키지 않고 데이터나 실험 난수를 재현합니다.
- **주의점:** 생성기를 다시 만들지 않으면 호출할 때마다 내부 상태가 진행됩니다.
- **공식 문서:** [np.random.default_rng](https://numpy.org/doc/stable/reference/random/generator.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# This tabular update is coupled to NumPy sampling, so it intentionally
# stays on CPU; neural replay variants move complete batches instead.
q = nn.Parameter(torch.zeros(n_states, n_actions))
opt = torch.optim.Adam([q], lr=0.05)
priority = torch.ones(len(rows))
rng = np.random.default_rng(442)
history = []
counts = np.zeros(len(rows), dtype=int)

portfolio_agent = PrioritizedReplayAgent(q, opt, priority, gamma)
initial_parameters = q.detach().clone()
for step in range(300):
    probability = portfolio_agent.sampling_probabilities().detach().numpy()
    index_numpy = rng.choice(
        len(rows),
        size=min(16, len(rows)),
        replace=True,
        p=probability,
    )
    counts += np.bincount(index_numpy, minlength=len(rows))
    index = torch.tensor(index_numpy)
    beta = 0.4 + 0.6 * step / 299
    history.append(portfolio_agent.update(index, beta))
parameter_delta = float((q.detach() - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()
assert priority.min() > 0

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§3.2` — **TD-error priority**
- **구현 이유:** 큰 surprise를 가진 transition을 더 자주 replay한다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.arange -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(q_star)
assert portfolio_metrics

probe_priority = torch.arange(1, 7, dtype=torch.float32)
expected = per_probabilities(probe_priority).cpu().numpy()
draws = np.random.default_rng(443).choice(6, size=5000, p=expected)
empirical = np.bincount(draws, minlength=6) / 5000
corr = float(np.corrcoef(expected, empirical)[0, 1])
q_mse = float(F.mse_loss(q.detach(), q_star))
print(f"sampling corr={corr:.3f}, Q MSE={q_mse:.4f}")
assert corr > 0.95 and math.isfinite(q_mse)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].bar(np.arange(6) - 0.18, expected, 0.36, label="Eq.(1)")
axes[0].bar(np.arange(6) + 0.18, empirical, 0.36, label="empirical")
axes[0].legend()
axes[1].plot(history)
axes[1].set_title("IS-weighted TD loss")
fig.tight_layout()
plt.show()

alpha=0은 uniform replay입니다. beta=1은 sampling 확률 변화에 대한 완전한 importance correction이며 논문은 학습 말기에 beta를 1로 anneal합니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3.2`의 **TD-error priority**
- **핵심 식:**

  $$
  P(i)=\frac{p_i^\alpha}{\sum_kp_k^\alpha},\qquad w_i=\left(NP(i)\right)^{-\beta}/\max_jw_j
  $$

- **입출력 shape:** `priority/probability/weight [N_replay], sampled indices [B], TD error [B]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `PrioritizedReplayAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. 큰 surprise를 가진 transition을 더 자주 replay한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.asarray -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.asarray(a, dtype=None, order=None)</code></summary>

#### `np.asarray(a, dtype=None, order=None)`

- **역할:** 가능하면 복사 없이 입력을 NumPy 배열로 해석합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.asarray](https://numpy.org/doc/stable/reference/generated/numpy.asarray.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 Atari replay trees와 Double DQN을 사용한다. 축소판은 모든 transition을 메모리에 두고 정확한 categorical probabilities를 계산한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.